# Thinking Out Loud — EEG EDA starter

Research direction: compare channel reduction for ML, DL, and pretrained models.
The eventual first model is an all-channel ML/SVM baseline. This notebook stops at **loading and EDA**;
it does not fit a classifier, select channels, or tune a model.

Run cells in order. The default loads **sub-01, ses-03**: a small session with 100 total trials
and 40 inner-speech trials before any exclusions. All 128 EEG channels are retained.
You can expand the subject/session lists later. A small session helps you learn the data;
its size is not intended to establish your final baseline performance.

The notebook is standalone. It can download the selected released EEG, events, and report files
into `DATA_ROOT` when missing (approximately 118 MB for the default session).
Set `DOWNLOAD_MISSING = False` to use files you already downloaded.

Data: [OpenNeuro ds003626, snapshot 2.1.2](https://openneuro.org/datasets/ds003626/versions/2.1.2).
Study: [Nieto et al., 2022](https://www.nature.com/articles/s41597-022-01147-2).


## 1. Packages

Use Python 3.10 or newer. If packages are missing, uncomment and run the installation line,
then restart your notebook kernel. `%pip` installs into the notebook's environment.
Scikit-learn can be added when we reach the ML baseline.


In [ ]:
# Uncomment only if you need to install the EDA packages:
# %pip install "mne>=1.6" numpy pandas matplotlib scipy


In [ ]:
from pathlib import Path
import hashlib
import re
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import scipy
from scipy.signal import welch
from IPython.display import display

mne.set_log_level("WARNING")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("MNE:", mne.__version__)
print("SciPy:", scipy.__version__)


## 2. Settings you can change

`DATA_ROOT` is the **dataset root**, not its `derivatives` folder. The expected layout is
`DATA_ROOT / derivatives / sub-01 / ses-03 / ...`.
Relative paths use your notebook kernel's current working directory. In Colab, you can use
`Path("/content/ds003626")`, or a mounted Drive directory.

`CONDITION = 1` selects inner speech. Keep `EXCLUDE_EMG = False` initially so you can inspect
the quality flags yourself. It is a report-based flag, not a guarantee that every artifact is detected.
The released epochs are already preprocessed; this starter adds no filters or rereferencing.


In [ ]:
DATA_ROOT = Path("./ds003626")
DATASET_VERSION = "2.1.2"
DOWNLOAD_MISSING = True

SUBJECTS = ["sub-01"]
SESSIONS = ["ses-03"]

# To expand later, for example:
# SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
# SESSIONS = ["ses-01", "ses-02", "ses-03"]
# All EEG derivatives are approximately 6.66 GB on disk.

CONDITION = 1
EXCLUDE_EMG = False
DATA_DTYPE = np.float32  # Store X in volts; float32 uses less RAM than float64.

CLASS_NAMES = {0: "up", 1: "down", 2: "right", 3: "left"}
SPANISH_WORDS = {0: "arriba", 1: "abajo", 2: "derecha", 3: "izquierda"}
CONDITION_NAMES = {0: "pronounced", 1: "inner", 2: "visualized"}

# These times are relative to cue onset, which is MNE time zero.
ACTION_WINDOW = (0.5, 3.0)

assert SUBJECTS and SESSIONS, "Choose at least one subject and session."
assert len(SUBJECTS) == len(set(SUBJECTS)), "Remove duplicate subjects."
assert len(SESSIONS) == len(set(SESSIONS)), "Remove duplicate sessions."
assert set(SUBJECTS).issubset({f"sub-{i:02d}" for i in range(1, 11)})
assert set(SESSIONS).issubset({"ses-01", "ses-02", "ses-03"})
assert CONDITION in CONDITION_NAMES

print("Dataset location:", DATA_ROOT.resolve())
print("Selected subjects:", SUBJECTS)
print("Selected sessions:", SESSIONS)
print("Selected condition:", CONDITION_NAMES[CONDITION])


## 3. Locate or download the selected files

The helper downloads only EEG epochs, labels and reports for your selected recordings.
It does not download raw BDF, baseline or EXG files. New downloads are checked against
the MD5 and file size recorded in snapshot 2.1.2. Existing ordinary files are reused.

GitHub stores annex pointers for the large files, so downloading their GitHub text alone does
not retrieve the EEG. This helper obtains the actual files from OpenNeuro's public mirror.
The event/report files are trusted dataset pickles; do not substitute unrelated pickle files.


In [ ]:
def ensure_dataset_file(relative_path):
    """Return a local file path, downloading a missing released file if enabled."""
    target = DATA_ROOT / relative_path

    if target.is_file():
        # Detect a Git-annex pointer downloaded as text instead of the real data.
        is_pointer = False
        if target.stat().st_size < 512:
            is_pointer = b".git/annex/objects" in target.read_bytes()
        if target.stat().st_size > 0 and not is_pointer:
            return target

    if not DOWNLOAD_MISSING:
        raise FileNotFoundError(
            f"Missing real dataset file: {target}. "
            "Check DATA_ROOT or set DOWNLOAD_MISSING = True."
        )

    pointer_url = (
        "https://raw.githubusercontent.com/OpenNeuroDatasets/ds003626/"
        f"{DATASET_VERSION}/{relative_path}"
    )
    with urllib.request.urlopen(pointer_url, timeout=60) as response:
        pointer_text = response.read().decode("utf-8")
    match = re.search(r"MD5E-s(\d+)--([0-9a-f]{32})", pointer_text)
    if match is None:
        raise ValueError(f"Cannot find the snapshot checksum for {relative_path}")
    expected_size = int(match.group(1))
    expected_md5 = match.group(2)

    mirror_url = "https://s3.amazonaws.com/openneuro.org/ds003626/" + relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    partial_path = target.with_name(target.name + ".part")
    checksum = hashlib.md5()
    print(f"Downloading {target.name} ({expected_size / 1e6:.1f} MB)")

    with urllib.request.urlopen(mirror_url, timeout=120) as response:
        with partial_path.open("wb") as output:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                checksum.update(chunk)

    if partial_path.stat().st_size != expected_size or checksum.hexdigest() != expected_md5:
        partial_path.unlink()
        raise ValueError(f"Snapshot checksum/size mismatch for {relative_path}")

    partial_path.replace(target)
    return target


recording_files = {}
for subject in SUBJECTS:
    for session in SESSIONS:
        prefix = f"derivatives/{subject}/{session}/{subject}_{session}"
        recording_files[(subject, session)] = {
            "eeg": ensure_dataset_file(prefix + "_eeg-epo.fif"),
            "events": ensure_dataset_file(prefix + "_events.dat"),
            "report": ensure_dataset_file(prefix + "_report.pkl"),
        }

print("Recordings ready:", len(recording_files))


## 4. Read labels and EEG together

Released event rows are `[raw_cue_sample, class_id, condition_id, session_number]`.
Their timestamps use the original recording's 1,024 Hz sample coordinates. The FIF epoch
event markers preserve these coordinates even though the saved signals are sampled at 256 Hz.

The loader checks event order before filtering. It retains the original trial index and an
`emg_flagged` column. Every selected EEG row stays aligned with its label and metadata.
It keeps all EEG channels, including any marked bad in the file, for initial inspection.


In [ ]:
def load_selected_eeg(recording_files):
    """Load the chosen condition from each recording, preserving row identities."""
    signal_parts = []
    selected_metadata_parts = []
    full_metadata_parts = []
    session_rows = []
    reference_info = None
    reference_times = None

    for (subject, session), paths in recording_files.items():
        events_array = np.asarray(pd.read_pickle(paths["events"]), dtype=np.int64)
        report = pd.read_pickle(paths["report"])
        session_epochs = mne.read_epochs(paths["eeg"], preload=False, proj=False, verbose="ERROR")

        if events_array.ndim != 2 or events_array.shape[1] != 4:
            raise ValueError(f"Unexpected event array shape: {subject}/{session}")
        if len(events_array) != len(session_epochs.events):
            raise ValueError(f"EEG/event row count mismatch: {subject}/{session}")
        if not np.array_equal(session_epochs.events[:, 0], events_array[:, 0]):
            raise ValueError(f"EEG/event timestamp order mismatch: {subject}/{session}")
        if not np.array_equal(session_epochs.events[:, 2], events_array[:, 1] + 31):
            raise ValueError(f"EEG/event class mismatch: {subject}/{session}")
        if not np.isin(events_array[:, 1], list(CLASS_NAMES)).all():
            raise ValueError("Unexpected class label.")
        if not np.isin(events_array[:, 2], list(CONDITION_NAMES)).all():
            raise ValueError("Unexpected condition label.")

        n_trials = len(events_array)
        flagged_indices = np.asarray(report.get("EMG_trials", []), dtype=int)
        if ((flagged_indices < 0) | (flagged_indices >= n_trials)).any():
            raise ValueError("Report contains an invalid EMG trial index.")
        emg_flagged = np.zeros(n_trials, dtype=bool)
        emg_flagged[flagged_indices] = True

        metadata_this_session = pd.DataFrame({
            "subject": subject,
            "session": session,
            "trial_index": np.arange(n_trials),
            "cue_sample_raw": events_array[:, 0],
            "class_id": events_array[:, 1],
            "condition_id": events_array[:, 2],
            "emg_flagged": emg_flagged,
        })
        metadata_this_session["class_name"] = metadata_this_session["class_id"].map(CLASS_NAMES)
        metadata_this_session["spanish_word"] = metadata_this_session["class_id"].map(SPANISH_WORDS)
        metadata_this_session["condition_name"] = metadata_this_session["condition_id"].map(CONDITION_NAMES)
        metadata_this_session["subject_session"] = subject + "/" + session
        metadata_this_session["trial_uid"] = (
            subject + "/" + session + "/" + metadata_this_session["trial_index"].astype(str)
        )
        full_metadata_parts.append(metadata_this_session)

        keep = events_array[:, 2] == CONDITION
        n_condition_trials = int(keep.sum())
        n_condition_flagged = int((keep & emg_flagged).sum())
        if EXCLUDE_EMG:
            keep = keep & ~emg_flagged
        selected_indices = np.flatnonzero(keep)

        session_rows.append({
            "subject": subject,
            "session": session,
            "total_trials": n_trials,
            "condition_trials_before_exclusion": n_condition_trials,
            "condition_emg_flagged": n_condition_flagged,
            "loaded_trials": len(selected_indices),
        })
        if len(selected_indices) == 0:
            continue

        eeg_picks = mne.pick_types(session_epochs.info, eeg=True, exclude=[])
        # Load only selected trials before changing the channel selection.
        session_epochs = session_epochs[selected_indices].load_data().pick(eeg_picks)
        if len(session_epochs) != len(selected_indices):
            raise ValueError("Unexpected trial rejection during loading; inspect row alignment.")

        if reference_info is None:
            reference_info = session_epochs.info.copy()
            reference_times = session_epochs.times.copy()
        else:
            if session_epochs.ch_names != reference_info["ch_names"]:
                raise ValueError("Channel names/order differ between recordings.")
            if session_epochs.info["sfreq"] != reference_info["sfreq"]:
                raise ValueError("Sampling rates differ between recordings.")
            if not np.array_equal(session_epochs.times, reference_times):
                raise ValueError("Epoch time coordinates differ between recordings.")

        signal_parts.append(session_epochs.get_data(copy=True).astype(DATA_DTYPE, copy=False))
        selected_metadata_parts.append(metadata_this_session.iloc[selected_indices].copy())
        print(f"Loaded {subject}/{session}: {len(selected_indices)} trials, {len(eeg_picks)} EEG channels")

    if not signal_parts:
        raise ValueError("No trials remain for the selected condition and artifact policy.")

    return {
        "X": np.concatenate(signal_parts, axis=0),
        "trial_metadata": pd.concat(selected_metadata_parts, ignore_index=True),
        "full_trial_metadata": pd.concat(full_metadata_parts, ignore_index=True),
        "session_summary": pd.DataFrame(session_rows),
        "info": reference_info,
        "times": reference_times,
    }


In [ ]:
loaded = load_selected_eeg(recording_files)

X = loaded["X"]
trial_metadata = loaded["trial_metadata"]
full_trial_metadata = loaded["full_trial_metadata"]
session_summary = loaded["session_summary"]
info = loaded["info"]
times = loaded["times"]

y = trial_metadata["class_id"].to_numpy(dtype=np.int64)
ch_names = list(info["ch_names"])
sfreq = float(info["sfreq"])
trial_times = times + 0.5  # Same samples, expressed relative to trial start.

# Keep these identities for future evaluation and grouped splitting.
groups_subject = trial_metadata["subject"].to_numpy()
groups_session = trial_metadata["subject_session"].to_numpy()
trial_ids = trial_metadata["trial_uid"].to_numpy()

assert X.shape[0] == len(y) == len(trial_metadata)
assert X.shape[1] == len(ch_names)
assert X.shape[2] == len(times)
assert trial_metadata["trial_uid"].is_unique

# Remove the duplicate dictionary reference; the variables above remain available.
del loaded

print("X shape [trials, channels, time points]:", X.shape)
print("y shape:", y.shape)
print("X units: volts; dtype:", X.dtype)
print("Sampling rate:", sfreq, "Hz")
print("MNE time range:", times[0], "to", times[-1], "seconds")
print("X memory:", round(X.nbytes / 1024**2, 1), "MiB")
display(session_summary)
display(trial_metadata.head())


## 5. Inspect metadata and class balance

`full_trial_metadata` contains all conditions in the recordings you selected.
`trial_metadata` contains only the rows actually present in `X` and `y`.
These tables describe your loaded subset; they are not automatically whole-dataset totals.


In [ ]:
trial_metadata.info()

print("Conditions in the selected recordings:")
display(full_trial_metadata["condition_name"].value_counts().rename("trials").to_frame())

class_counts = pd.Series(y).value_counts().reindex(list(CLASS_NAMES), fill_value=0).sort_index()
class_balance = pd.DataFrame({
    "class_id": class_counts.index,
    "class_name": [CLASS_NAMES[i] for i in class_counts.index],
    "trials": class_counts.to_numpy(),
})
class_balance["percent"] = 100 * class_balance["trials"] / len(y)
display(class_balance)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_balance["class_name"], class_balance["trials"], color="#3269ce")
ax.set(xlabel="Direction", ylabel="Loaded trials", title="Class balance in the loaded EEG")
plt.tight_layout()
plt.show()

class_counts_by_session = pd.crosstab(
    trial_metadata["subject_session"], trial_metadata["class_name"]
).reindex(columns=list(CLASS_NAMES.values()), fill_value=0)
display(class_counts_by_session)


## 6. Inspect the EMG flags without changing the data

The counts below use the original condition rows, before your exclusion setting.
With `EXCLUDE_EMG = False`, flagged trials remain available for inspection.
Changing this policy later requires rerunning the loading cell and all downstream EDA.


In [ ]:
condition_metadata = full_trial_metadata.loc[
    full_trial_metadata["condition_id"] == CONDITION
].copy()

emg_summary = condition_metadata.groupby("subject_session").agg(
    condition_trials=("trial_uid", "size"),
    emg_flagged=("emg_flagged", "sum"),
)
emg_summary["flagged_percent"] = 100 * emg_summary["emg_flagged"] / emg_summary["condition_trials"]
display(emg_summary)

print("Flagged trials currently present in X:", int(trial_metadata["emg_flagged"].sum()))
display(trial_metadata.loc[trial_metadata["emg_flagged"]].head())


## 7. Basic signal checks and channel statistics

Arrays stay in volts. Convert to microvolts only for descriptive statistics and plotting.
The checks report nonfinite values and amplitude variation; they do not automatically
reject trials or channels. A channel with large amplitude is a reason to inspect it, not
an automatic label of a bad channel.


In [ ]:
# Check one trial at a time to avoid allocating a large whole-dataset boolean tensor.
nonfinite_counts = np.array([np.count_nonzero(~np.isfinite(trial)) for trial in X])
print("Trials containing NaN/inf:", int((nonfinite_counts > 0).sum()))
print("Total NaN/inf values:", int(nonfinite_counts.sum()))
print("Channels marked bad in the first recording:", info["bads"])

trial_channel_std = X.std(axis=-1)  # [trials, channels], in volts
trial_channel_ptp = np.ptp(X, axis=-1)

channel_summary = pd.DataFrame({
    "channel_index": np.arange(len(ch_names)),
    "channel_name": ch_names,
    "median_trial_std_uV": np.median(trial_channel_std, axis=0) * 1e6,
    "median_trial_ptp_uV": np.median(trial_channel_ptp, axis=0) * 1e6,
})
display(channel_summary.head())
display(channel_summary.sort_values("median_trial_std_uV", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(channel_summary["channel_index"], channel_summary["median_trial_std_uV"])
ax.set(xlabel="Channel index in X", ylabel="Median trial SD (µV)", title="Amplitude variation across all loaded channels")
plt.tight_layout()
plt.show()


## 8. Channel names and electrode positions

`ch_names[i]` identifies channel axis `i` in `X`.
`channel_table` records positions in meters. The released file may lack positions.
In that case, use the standard BioSemi 128 layout and label its source explicitly.
These template positions support visualization; they are not measured individual head coordinates.


In [ ]:
plot_info = info.copy()
channel_positions = np.array([channel["loc"][:3] for channel in plot_info["chs"]])
positions_available = (
    np.isfinite(channel_positions).all()
    and (np.linalg.norm(channel_positions, axis=1) > 0).all()
)
montage_source = "file"
if not positions_available:
    standard_montage = mne.channels.make_standard_montage("biosemi128")
    if set(ch_names).issubset(set(standard_montage.ch_names)):
        plot_info.set_montage(standard_montage, on_missing="raise")
        channel_positions = np.array([channel["loc"][:3] for channel in plot_info["chs"]])
        positions_available = True
        montage_source = "standard_biosemi128_template"
    else:
        montage_source = "unavailable"

channel_table = pd.DataFrame({
    "channel_index": np.arange(len(ch_names)),
    "channel_name": ch_names,
    "x_m": channel_positions[:, 0],
    "y_m": channel_positions[:, 1],
    "z_m": channel_positions[:, 2],
    "position_source": montage_source,
})
display(channel_table.head(12))
print("Position source:", montage_source)
if positions_available:
    mne.viz.plot_sensors(plot_info, kind="topomap", show_names=False, show=False)
    plt.show()
else:
    print("Valid positions are missing for some channels; no sensor layout is plotted.")


## 9. Plot one trial

`trial_index` below indexes the **loaded X array**. The `trial_index` column in metadata
is the original index within its session; these indices differ after condition selection.
Change `trial_index` and `channels_to_plot` to explore.

The x-axis uses MNE time: cue 0–0.5 s, action 0.5–3.0 s. Array sample zero corresponds
to MNE time −0.5 s and trial time 0 s.


In [ ]:
trial_index = 0
channels_to_plot = [name for name in ["A1", "B1", "C1", "D1"] if name in ch_names]
channel_indices = [ch_names.index(name) for name in channels_to_plot]

assert 0 <= trial_index < len(X)
assert channel_indices, "Choose at least one channel present in ch_names."
display(trial_metadata.iloc[[trial_index]])

fig, axes = plt.subplots(len(channel_indices), 1, figsize=(11, 6), sharex=True, squeeze=False)
for ax, channel_index in zip(axes.ravel(), channel_indices):
    ax.plot(times, X[trial_index, channel_index] * 1e6, linewidth=0.8)
    ax.axvspan(0.0, 0.5, color="orange", alpha=0.15, label="Cue")
    ax.axvspan(*ACTION_WINDOW, color="#3269ce", alpha=0.10, label="Action")
    ax.set_ylabel(ch_names[channel_index] + " (µV)")
axes[0, 0].legend(loc="upper right", ncol=2)
axes[-1, 0].set_xlabel("Seconds relative to cue onset")
fig.suptitle(f"Loaded trial {trial_index}: {CLASS_NAMES[int(y[trial_index])]}")
plt.tight_layout()
plt.show()


## 10. Inspect all channels in the same trial

The color map shows amplitude, not the probability of a class or a neural-source location.
Its display range uses a percentile so a few extremes do not dominate the image.


In [ ]:
trial_uV = X[trial_index] * 1e6
color_limit = max(float(np.nanpercentile(np.abs(trial_uV), 99)), 1e-6)

fig, ax = plt.subplots(figsize=(11, 6))
image = ax.imshow(
    trial_uV,
    aspect="auto",
    origin="lower",
    extent=[times[0], times[-1], -0.5, len(ch_names) - 0.5],
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
)
ticks = np.arange(0, len(ch_names), 16)
ax.set_yticks(ticks)
ax.set_yticklabels([ch_names[i] for i in ticks])
ax.axvline(ACTION_WINDOW[0], color="black", linestyle="--", linewidth=0.8)
ax.axvline(ACTION_WINDOW[1], color="black", linestyle="--", linewidth=0.8)
ax.set(xlabel="Seconds relative to cue onset", ylabel="EEG channel", title=f"All channels, loaded trial {trial_index}")
ax.grid(False)
fig.colorbar(image, ax=ax, label="Amplitude (µV)")
plt.tight_layout()
plt.show()


## 11. Explore power spectra during action

Welch PSD estimates power at each frequency. The class curves below average over trials
and channels in your selected recordings. They are descriptive plots, not validation results.
Class differences in a small session may be unstable.

`X` retains the full saved epoch. `action_mask` selects the action samples;
`psd` has axes `[trials, channels, frequencies]` and units V²/Hz.
For all participants, this computation may use substantial RAM. Expand gradually.


In [ ]:
action_mask = (times >= ACTION_WINDOW[0]) & (times <= ACTION_WINDOW[1])
assert action_mask.sum() >= 2, "The requested action window contains too few samples."
assert np.isfinite(X).all(), "Inspect and handle NaN/inf before spectral analysis."

nperseg = min(int(sfreq), int(action_mask.sum()))  # About one second per Welch segment.
freqs, psd = welch(
    X[:, :, action_mask],
    fs=sfreq,
    nperseg=nperseg,
    noverlap=nperseg // 2,
    axis=-1,
    scaling="density",
)
print("PSD shape [trials, channels, frequencies]:", psd.shape)

frequency_mask = (freqs >= 1) & (freqs <= min(100, sfreq / 2))
fig, ax = plt.subplots(figsize=(9, 4))
for class_id, class_name in CLASS_NAMES.items():
    if not (y == class_id).any():
        continue
    mean_psd_uV2 = psd[y == class_id].mean(axis=(0, 1)) * 1e12
    ax.plot(
        freqs[frequency_mask],
        10 * np.log10(np.maximum(mean_psd_uV2[frequency_mask], 1e-30)),
        label=class_name,
    )
ax.set(xlabel="Frequency (Hz)", ylabel="PSD (dB relative to 1 µV²/Hz)", title="Action-window spectra by class")
ax.legend()
plt.tight_layout()
plt.show()


## 12. Explore correlation between channels

This correlation matrix uses **one trial's action interval**. Shared activity may reflect
neural signals, volume conduction, reference effects or artifacts. High correlation alone
does not show that a channel can be removed without changing classification performance.
Any future channel selection must be estimated using training data within the evaluation split.


In [ ]:
# Index the trial first, then its time axis, to preserve [channels, samples].
trial_action = X[trial_index][:, action_mask]
channel_correlation = np.corrcoef(trial_action)

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(channel_correlation, vmin=-1, vmax=1, cmap="RdBu_r", origin="lower")
ticks = np.arange(0, len(ch_names), 16)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels([ch_names[i] for i in ticks], rotation=90)
ax.set_yticklabels([ch_names[i] for i in ticks])
ax.set(xlabel="Channel", ylabel="Channel", title=f"Action-window correlation, loaded trial {trial_index}")
ax.grid(False)
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

upper_triangle = channel_correlation[np.triu_indices(len(ch_names), k=1)]
print("Median absolute pairwise correlation in this trial:", np.nanmedian(np.abs(upper_triangle)))


## Variables we will reuse in future code

Future snippets in this conversation will build on these names. If you rename a variable or
change a preprocessing step, tell me or paste that cell so the next snippet matches your state.

| Variable | Meaning |
|---|---|
| `DATA_ROOT` | Local root of ds003626 |
| `SUBJECTS`, `SESSIONS` | Recordings chosen for loading |
| `CONDITION`, `EXCLUDE_EMG` | Task selection and artifact policy |
| `X` | EEG in volts, `[trials, channels, time points]`; full saved epoch |
| `y` | Integer direction labels, `[trials]` |
| `trial_metadata` | One metadata row per row of `X`, in the same order |
| `full_trial_metadata` | All conditions in your selected recordings, before selection |
| `session_summary` | Counts for each selected recording |
| `ch_names` | Channel names in the order of `X`'s channel axis |
| `sfreq` | Sampling frequency in Hz |
| `times` | Time coordinates relative to cue onset |
| `trial_times` | Same time coordinates relative to trial start |
| `info` | MNE measurement info from the first recording |
| `channel_table` | Channel indices, names, coordinates and their source |
| `plot_info` | Info copy with a standard BioSemi layout when file positions are missing |
| `channel_summary` | Descriptive amplitude statistics per channel |
| `groups_subject` | Subject identity per trial |
| `groups_session` | Subject/session identity per trial |
| `trial_ids` | Unique original trial identity per trial |
| `ACTION_WINDOW`, `action_mask` | Action interval and its time-axis selection |
| `freqs`, `psd` | Welch frequency bins and action-window spectral estimates |
| `trial_index` | A row index into the loaded `X` for plotting |
| `channel_correlation` | Correlation matrix for the example trial's action interval |

## Before the later baseline

Keep the first baseline at all 128 channels. Decide whether you want within-person,
held-out-session, or held-out-person evaluation before choosing a split. With only the default
session, there is just one session group; session-held-out evaluation requires additional sessions.
Four-direction balanced-accuracy chance level is 25%.

When we add SVM/ML code later, fit scaling, feature selection and tuning inside training folds.
For the eventual ML/DL/pretrained comparison, keep trial splits, artifact policy and timing
consistent, and document model-specific preprocessing. Pretrained-model channel layouts and
sampling requirements will need checking for the chosen model.

This notebook does not reserve a test set yet. EDA here is practice on a small development subset;
decide which recordings will remain untouched for final evaluation before using plots to tune choices.

References for the APIs used: [MNE read_epochs](https://mne.tools/stable/generated/mne.read_epochs.html),
[MNE Epochs.get_data](https://mne.tools/stable/generated/mne.Epochs.html#mne.Epochs.get_data),
[SciPy Welch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html).


In [ ]:
# Your own exploration starts here.
# Future code snippets will reuse X, y, trial_metadata, ch_names, sfreq and times.
